# MethylGPT: Extract Cell-Level Embeddings

This notebook demonstrates how to extract dense sample-level (cell) embeddings from a pretrained MethylGPT model. These embeddings can be used for:

- UMAP visualization and clustering
- Downstream classification and regression tasks
- Batch effect analysis

**Requirements:**
- Pretrained MethylGPT model checkpoint
- Preprocessed Parquet data files
- CpG probe IDs CSV

In [ ]:
# Colab / environment setup
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q methylgpt[tutorials]
    !pip install -q gdown
    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING: No GPU. Go to Runtime > Change runtime type > GPU")

In [ ]:
import os
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import torch

from methylgpt import MethylGPTModel, MethylVocab, create_dataloader
from methylgpt.inference import extract_embeddings

warnings.filterwarnings("ignore", message=".*IProgress.*")
warnings.filterwarnings("ignore", message=".*flash_attn.*")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Configure Paths

Update these paths to point to your local files. See the [README](../../README.md#pretrained-models) for download links.

In [ ]:
# === UPDATE THESE PATHS ===
# Parquet data directory
PARQUET_DIR = "data/processed_type3_parquet_shuffled"
# Pretrained model directory (containing args.json and model .pt file)
MODEL_DIR = "pretrained_models/methylgpt-medium"
# CpG probe IDs file
CPG_LIST_FILE = "data/probe_ids_type3.csv"
# Output directory for embeddings
SAVE_DIR = Path("Embeddings")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# If running on Colab, download all required files
if IN_COLAB:
    import gdown, subprocess
    os.makedirs("data", exist_ok=True)

    # 1. Download pretrained model (medium)
    if not list(Path(MODEL_DIR).glob("*.pt")) if Path(MODEL_DIR).exists() else True:
        os.makedirs(MODEL_DIR, exist_ok=True)
        print("Downloading methylgpt-medium model...")
        gdown.download_folder(
            "https://drive.google.com/drive/folders/14M4wdS83el9PAgh9TdfjSCeEcDPbz34f",
            output=MODEL_DIR, quiet=True
        )
        print(f"Model downloaded to {MODEL_DIR}")
    else:
        print(f"Model already exists in {MODEL_DIR}")

    # 2. Download probe_ids_type3.csv
    if not os.path.exists(CPG_LIST_FILE):
        print("Downloading probe_ids_type3.csv...")
        subprocess.run([
            "wget", "-q", "-O", CPG_LIST_FILE,
            "https://www.dropbox.com/scl/fi/2n6bx7j8v0aon0kwfsghp/probe_ids_type3.csv?rlkey=ly133xlce1xxjiku6tiski6qq&st=pig4e41h&dl=1"
        ], check=True)
        print(f"Probe IDs saved to {CPG_LIST_FILE}")
    else:
        print(f"Probe IDs already exist at {CPG_LIST_FILE}")

    # 3. Download sample parquet data
    if not os.path.exists(PARQUET_DIR):
        print("Downloading sample parquet data (~2 GB)...")
        subprocess.run([
            "wget", "-q", "--show-progress", "-O", "data/parquet_data.tar.gz",
            "https://www.dropbox.com/scl/fi/bbs6sxlkpbx11rhyvdfto/processed_type3_parquet_shuffled.tar.gz?rlkey=s73utmumq6xldmv3y6kh9bz75&st=8pslwy2a&dl=1"
        ], check=True)
        subprocess.run(["tar", "-xzf", "data/parquet_data.tar.gz", "-C", "data/"], check=True)
        os.remove("data/parquet_data.tar.gz")
        print(f"Parquet data extracted to {PARQUET_DIR}")
    else:
        print(f"Parquet data already exists at {PARQUET_DIR}")


## 2. Load Model

In [ ]:
# Load model config
with open(Path(MODEL_DIR) / "args.json", "r") as f:
    config = json.load(f)

# Find model checkpoint file
model_files = list(Path(MODEL_DIR).glob("*.pt"))
assert len(model_files) > 0, f"No .pt files found in {MODEL_DIR}"
model_file = str(model_files[0])
print(f"Using model checkpoint: {model_file}")

# Update config for inference
config["load_model"] = True
config["pretrained_file"] = model_file
config["mask_ratio"] = 0  # No masking for embedding extraction
config["probe_id_dir"] = CPG_LIST_FILE

# Load vocabulary and model
vocab = MethylVocab(
    probe_id_dir=CPG_LIST_FILE,
    pad_token="<pad>",
    special_tokens=["<pad>", "<cls>", "<eoc>"],
    save_dir=None,
)

model = MethylGPTModel.from_pretrained(config, vocab)
model.eval()
model.to(device)

# Use half precision on GPU for speed
if device.type == "cuda":
    model.half()

print(f"Model loaded: {config['layer_size']}-dim, {config['nlayers']} layers")

## 3. Create Data Loader

In [ ]:
# List Parquet files
parquet_files = sorted([
    os.path.join(PARQUET_DIR, f) for f in os.listdir(PARQUET_DIR)
    if f.endswith(".parquet")
])
print(f"Found {len(parquet_files)} Parquet files")

# Create data loader (use first file for demo, or all files for full extraction)
data_loader = create_dataloader([parquet_files[0]], batch_size=config.get("batch_size", 32))

## 4. Extract Embeddings

In [ ]:
# Extract embeddings using the high-level API
# Set max_batches to limit processing for demo (remove for full extraction)
embeddings, sample_ids = extract_embeddings(
    model, data_loader, device=str(device), max_batches=100
)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Number of samples: {len(sample_ids)}")

## 5. Save Embeddings

In [ ]:
# Save embeddings
output_path = SAVE_DIR / "cell_embeddings.pkl"
with open(output_path, "wb") as f:
    pickle.dump({"cell_emb": embeddings, "cell_list": sample_ids}, f)

print(f"Embeddings saved to {output_path}")

## 6. Quick Visualization (UMAP)

Requires `umap-learn` and `matplotlib` (install with `pip install methylgpt[tutorials]`).

In [ ]:
try:
    import umap
    import matplotlib.pyplot as plt
    from aquarel import load_theme

    theme = (
        load_theme("scientific")
        .set_grid(draw=False)
        .set_font(size=15)
        .set_ticks(direction="out")
        .set_axis_labels(pad=10)
    )
    theme.apply()

    # Compute UMAP
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    umap_coords = reducer.fit_transform(embeddings)

    # Plot with outline layer
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(umap_coords[:, 0], umap_coords[:, 1], s=8, c="black", alpha=1, zorder=1)
    ax.scatter(umap_coords[:, 0], umap_coords[:, 1], s=5, alpha=0.5, c="steelblue", zorder=2)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.set_title("MethylGPT Embeddings (UMAP)")

    theme.apply_transforms()

    plt.savefig(str(SAVE_DIR / "umap_embeddings.pdf"), bbox_inches="tight")
    plt.savefig(str(SAVE_DIR / "umap_embeddings.png"), dpi=600, bbox_inches="tight")
    plt.show()
    print("UMAP saved to Embeddings/umap_embeddings.pdf")

except ImportError:
    print("Install visualization deps: pip install methylgpt[tutorials]")

## Next Steps

- **Color UMAP by metadata**: Load metadata and color by tissue, age, sex, etc. See [Embedding Analysis tutorial](../embedding_analysis/)
- **Downstream tasks**: Use embeddings as features for classification/regression. See [Disease Prediction tutorial](../disease_prediction/)
- **Full dataset**: Remove `max_batches` limit and process all Parquet files
- **CpG embeddings**: Use `extract_cpg_embeddings()` for CpG-level analysis